In [ ]:
import json
import os
import time
from pathlib import Path

import torch
from tqdm.auto import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer


# CONFIG 
DATA_DIR = Path(os.environ.get("DATA_DIR", "/kaggle/input/datasets/anisoaraana/similarity"))
OUT_DIR = Path(os.environ.get("OUT_DIR", "/kaggle/working/romanian_nllb_fast"))
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = os.environ.get("TRANSLATION_MODEL", "facebook/nllb-200-distilled-600M")

SRC_LANG = "eng_Latn"
TGT_LANG = "ron_Latn"

BATCH_SIZE = int(os.environ.get("TRANSLATE_BS", 8))
MAX_INPUT_TOKENS = int(os.environ.get("MAX_INPUT_TOKENS", 512))
MAX_NEW_TOKENS = int(os.environ.get("MAX_NEW_TOKENS", 512))

CACHE_PATH = OUT_DIR / "translation_cache_en_ro.json"
MANIFEST_PATH = OUT_DIR / "translation_manifest.json"

FILES = {
    "synthetic_data_for_classification.jsonl": ["anchor_text", "text_a", "text_b"],
    "synthetic_data_new.jsonl": ["anchor_text", "text_a", "text_b"],
    "dev_track_a.jsonl": ["anchor_text", "text_a", "text_b"],
    "test_track_a.jsonl": ["anchor_text", "text_a", "text_b"],
    "test_track_b.jsonl": ["text"],
}


# HELPERS
def norm_text(text):
    return " ".join(str(text).split())


def read_jsonl(path):
    rows = []
    if not path.exists():
        return rows
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def write_jsonl(path, rows):
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def load_cache(path):
    if not path.exists():
        return {}
    with path.open(encoding="utf-8") as f:
        return json.load(f)


def save_cache(path, cache):
    tmp_path = path.with_suffix(".tmp")
    with tmp_path.open("w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False)
    tmp_path.replace(path)


def collect_unique_texts():
    seen = set()
    texts = []
    file_stats = {}

    for filename, fields in FILES.items():
        path = DATA_DIR / filename
        rows = read_jsonl(path)
        if not rows:
            file_stats[filename] = {"exists": False, "rows": 0, "fields": fields}
            continue

        n_fields = 0
        for row in rows:
            for field in fields:
                value = norm_text(row.get(field, ""))
                if not value:
                    continue
                n_fields += 1
                if value not in seen:
                    seen.add(value)
                    texts.append(value)

        file_stats[filename] = {
            "exists": True,
            "rows": len(rows),
            "text_fields_seen": n_fields,
            "fields": fields,
        }

    return texts, file_stats


# TRANSLATION 
def load_model():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device}")
    print(f"Translation model: {MODEL_NAME}")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
    model.to(device)
    model.eval()

    if device == "cuda":
        model.half()
        torch.backends.cuda.matmul.allow_tf32 = True

    tokenizer.src_lang = SRC_LANG
    forced_bos_token_id = tokenizer.convert_tokens_to_ids(TGT_LANG)
    return tokenizer, model, device, forced_bos_token_id


def translate_batch(tokenizer, model, device, forced_bos_token_id, batch_texts):
    inputs = tokenizer(
        batch_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
    ).to(device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            forced_bos_token_id=forced_bos_token_id,
            max_new_tokens=MAX_NEW_TOKENS,
            num_beams=1,
            do_sample=False,
            use_cache=True,
        )

    return tokenizer.batch_decode(outputs, skip_special_tokens=True)


def translate_all(texts):
    cache = load_cache(CACHE_PATH)
    missing = [text for text in texts if text not in cache]
    print(f"Unique source texts: {len(texts)}")
    print(f"Already cached: {len(texts) - len(missing)}")
    print(f"To translate now: {len(missing)}")

    if not missing:
        return cache

    tokenizer, model, device, forced_bos_token_id = load_model()
    start = time.time()

    for start_idx in tqdm(range(0, len(missing), BATCH_SIZE), desc="Translating en->ro"):
        batch = missing[start_idx : start_idx + BATCH_SIZE]
        translations = translate_batch(tokenizer, model, device, forced_bos_token_id, batch)
        for src, tgt in zip(batch, translations):
            cache[src] = norm_text(tgt)

        # Save often so a Kaggle interruption can resume.
        if (start_idx // BATCH_SIZE) % 25 == 0:
            save_cache(CACHE_PATH, cache)

    save_cache(CACHE_PATH, cache)
    elapsed = time.time() - start
    print(f"Translated {len(missing)} texts in {elapsed / 60:.1f} minutes")
    return cache


# WRITE ROMANIAN DATASETS 
def write_translated_files(cache):
    outputs = {}
    for filename, fields in FILES.items():
        in_path = DATA_DIR / filename
        rows = read_jsonl(in_path)
        if not rows:
            continue

        translated_rows = []
        for row in rows:
            out = dict(row)
            for field in fields:
                src = norm_text(out.get(field, ""))
                if src:
                    out[field] = cache[src]
            translated_rows.append(out)

        out_path = OUT_DIR / filename.replace(".jsonl", "_ro.jsonl")
        write_jsonl(out_path, translated_rows)
        outputs[filename] = str(out_path)
        print(f"Wrote {out_path} ({len(translated_rows)} rows)")

    return outputs


def main():
    texts, file_stats = collect_unique_texts()
    cache = translate_all(texts)
    outputs = write_translated_files(cache)

    manifest = {
        "model": MODEL_NAME,
        "source_language": SRC_LANG,
        "target_language": TGT_LANG,
        "batch_size": BATCH_SIZE,
        "max_input_tokens": MAX_INPUT_TOKENS,
        "max_new_tokens": MAX_NEW_TOKENS,
        "num_beams": 1,
        "do_sample": False,
        "cache_path": str(CACHE_PATH),
        "data_dir": str(DATA_DIR),
        "out_dir": str(OUT_DIR),
        "file_stats": file_stats,
        "outputs": outputs,
        "unique_texts": len(texts),
        "cached_translations": len(cache),
    }
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"Saved manifest: {MANIFEST_PATH}")


if __name__ == "__main__":
    main()


In [4]:
import os
os.environ["MAX_CHUNK_WORDS"] = "50"
os.environ["FORCE_RETRANSLATE"] = "1"

In [ ]:
# This version is safer than one-shot story translation:
# - deduplicates story texts;
# - splits long summaries into sentence chunks;
# - uses anti-repetition generation settings;
# - writes reusable story/chunk caches;
# - flags suspicious translations for manual inspection.

import json
import os
import re
import time
from pathlib import Path

import torch
from tqdm.auto import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer


# CONFIG
DATA_DIR = Path(os.environ.get("DATA_DIR", "/kaggle/input/datasets/anisoaraana/similarity"))
OUT_DIR = Path(os.environ.get("OUT_DIR", "/kaggle/working/romanian_nllb_improved"))
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = os.environ.get("TRANSLATION_MODEL", "facebook/nllb-200-distilled-600M")

SRC_LANG = "eng_Latn"
TGT_LANG = "ron_Latn"

BATCH_SIZE = int(os.environ.get("TRANSLATE_BS", 16))
MAX_INPUT_TOKENS = int(os.environ.get("MAX_INPUT_TOKENS", 256))
MAX_NEW_TOKENS = int(os.environ.get("MAX_NEW_TOKENS", 320))
MAX_CHUNK_WORDS = int(os.environ.get("MAX_CHUNK_WORDS", 80))
NUM_BEAMS = int(os.environ.get("NUM_BEAMS", 2))
NO_REPEAT_NGRAM_SIZE = int(os.environ.get("NO_REPEAT_NGRAM_SIZE", 4))
REPETITION_PENALTY = float(os.environ.get("REPETITION_PENALTY", 1.12))
LENGTH_PENALTY = float(os.environ.get("LENGTH_PENALTY", 1.0))
FORCE_RETRANSLATE = os.environ.get("FORCE_RETRANSLATE", "0").strip().lower() in {"1", "true", "yes"}

CACHE_PATH = OUT_DIR / "translation_cache_en_ro.json"
CHUNK_CACHE_PATH = OUT_DIR / "translation_chunk_cache_en_ro.json"
MANIFEST_PATH = OUT_DIR / "translation_manifest.json"
QUALITY_REPORT_PATH = OUT_DIR / "translation_quality_report.json"

FILES = {
    "synthetic_data_for_classification.jsonl": ["anchor_text", "text_a", "text_b"],
    "synthetic_data_new.jsonl": ["anchor_text", "text_a", "text_b"],
    "dev_track_a.jsonl": ["anchor_text", "text_a", "text_b"],
    "test_track_a.jsonl": ["anchor_text", "text_a", "text_b"],
    "test_track_b.jsonl": ["text"],
}


# HELPERS
def norm_text(text):
    return " ".join(str(text).split())


def split_sentences(text):
    text = norm_text(text)
    if not text:
        return []
    # Narrative summaries are mostly plain prose. This keeps the splitter simple
    # and avoids extra dependencies such as spaCy/NLTK in Kaggle.
    parts = re.split(r"(?<=[.!?])\s+(?=[A-Z\"(])", text)
    return [p.strip() for p in parts if p.strip()]


def chunk_text(text, max_words=MAX_CHUNK_WORDS):
    sentences = split_sentences(text)
    if not sentences:
        return []

    chunks = []
    current = []
    current_words = 0

    for sentence in sentences:
        words = sentence.split()
        if len(words) > max_words:
            if current:
                chunks.append(" ".join(current))
                current = []
                current_words = 0
            for i in range(0, len(words), max_words):
                chunks.append(" ".join(words[i : i + max_words]))
            continue

        if current and current_words + len(words) > max_words:
            chunks.append(" ".join(current))
            current = [sentence]
            current_words = len(words)
        else:
            current.append(sentence)
            current_words += len(words)

    if current:
        chunks.append(" ".join(current))
    return chunks


def has_degenerate_repetition(text):
    words = re.findall(r"\w+", text.lower())
    if len(words) < 12:
        return False
    for n in (1, 2, 3, 4):
        for i in range(0, len(words) - 3 * n + 1):
            gram = words[i : i + n]
            if words[i + n : i + 2 * n] == gram and words[i + 2 * n : i + 3 * n] == gram:
                return True
    return False


def quality_flags(src, tgt):
    src_words = max(len(src.split()), 1)
    tgt_words = len(tgt.split())
    ratio = tgt_words / src_words
    flags = []
    if ratio < 0.60:
        flags.append("too_short_or_truncated")
    if ratio > 1.55:
        flags.append("too_long_or_run_on")
    if has_degenerate_repetition(tgt):
        flags.append("repetition")
    if tgt.count("...") > 1:
        flags.append("ellipsis_artifact")
    if len(tgt.strip()) < 20 and len(src.strip()) > 80:
        flags.append("near_empty")
    return flags, ratio


def read_jsonl(path):
    rows = []
    if not path.exists():
        return rows
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def write_jsonl(path, rows):
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def load_cache(path):
    if not path.exists():
        return {}
    with path.open(encoding="utf-8") as f:
        return json.load(f)


def save_cache(path, cache):
    tmp_path = path.with_suffix(".tmp")
    with tmp_path.open("w", encoding="utf-8") as f:
        json.dump(cache, f, ensure_ascii=False)
    tmp_path.replace(path)


def collect_unique_texts():
    seen = set()
    texts = []
    file_stats = {}

    for filename, fields in FILES.items():
        path = DATA_DIR / filename
        rows = read_jsonl(path)
        if not rows:
            file_stats[filename] = {"exists": False, "rows": 0, "fields": fields}
            continue

        n_fields = 0
        for row in rows:
            for field in fields:
                value = norm_text(row.get(field, ""))
                if not value:
                    continue
                n_fields += 1
                if value not in seen:
                    seen.add(value)
                    texts.append(value)

        file_stats[filename] = {
            "exists": True,
            "rows": len(rows),
            "text_fields_seen": n_fields,
            "fields": fields,
        }

    return texts, file_stats


# TRANSLATION 
def load_model():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Device: {device}")
    print(f"Translation model: {MODEL_NAME}")

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
    model.to(device)
    model.eval()

    if device == "cuda":
        model.half()
        torch.backends.cuda.matmul.allow_tf32 = True

    tokenizer.src_lang = SRC_LANG
    if hasattr(tokenizer, "lang_code_to_id") and TGT_LANG in tokenizer.lang_code_to_id:
        forced_bos_token_id = tokenizer.lang_code_to_id[TGT_LANG]
    else:
        forced_bos_token_id = tokenizer.convert_tokens_to_ids(TGT_LANG)
    return tokenizer, model, device, forced_bos_token_id


def translate_batch(tokenizer, model, device, forced_bos_token_id, batch_texts):
    inputs = tokenizer(
        batch_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
    ).to(device)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            forced_bos_token_id=forced_bos_token_id,
            max_new_tokens=MAX_NEW_TOKENS,
            num_beams=NUM_BEAMS,
            do_sample=False,
            no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
            repetition_penalty=REPETITION_PENALTY,
            length_penalty=LENGTH_PENALTY,
            early_stopping=NUM_BEAMS > 1,
            use_cache=True,
        )

    return tokenizer.batch_decode(outputs, skip_special_tokens=True)


def translate_all(texts):
    story_cache = {} if FORCE_RETRANSLATE else load_cache(CACHE_PATH)
    chunk_cache = {} if FORCE_RETRANSLATE else load_cache(CHUNK_CACHE_PATH)

    story_chunks = {text: chunk_text(text) for text in texts}
    missing_stories = [text for text in texts if text not in story_cache]
    unique_chunks = []
    seen_chunks = set()

    for text in missing_stories:
        for chunk in story_chunks[text]:
            if chunk not in chunk_cache and chunk not in seen_chunks:
                seen_chunks.add(chunk)
                unique_chunks.append(chunk)

    print(f"Unique source stories: {len(texts)}")
    print(f"Already cached stories: {len(texts) - len(missing_stories)}")
    print(f"Stories to assemble now: {len(missing_stories)}")
    print(f"Chunk cache size: {len(chunk_cache)}")
    print(f"Chunks to translate now: {len(unique_chunks)}")

    if unique_chunks:
        tokenizer, model, device, forced_bos_token_id = load_model()
        start = time.time()

        for start_idx in tqdm(range(0, len(unique_chunks), BATCH_SIZE), desc="Translating chunks en->ro"):
            batch = unique_chunks[start_idx : start_idx + BATCH_SIZE]
            translations = translate_batch(tokenizer, model, device, forced_bos_token_id, batch)
            for src, tgt in zip(batch, translations):
                chunk_cache[src] = norm_text(tgt)

            # Save often so a Kaggle interruption can resume.
            if (start_idx // BATCH_SIZE) % 25 == 0:
                save_cache(CHUNK_CACHE_PATH, chunk_cache)

        save_cache(CHUNK_CACHE_PATH, chunk_cache)
        elapsed = time.time() - start
        print(f"Translated {len(unique_chunks)} chunks in {elapsed / 60:.1f} minutes")

    for text in missing_stories:
        chunks = story_chunks[text]
        translated_chunks = [chunk_cache[ch] for ch in chunks if ch in chunk_cache]
        story_cache[text] = norm_text(" ".join(translated_chunks))

    save_cache(CACHE_PATH, story_cache)
    return story_cache


# WRITE ROMANIAN DATASETS 
def write_translated_files(cache):
    outputs = {}
    for filename, fields in FILES.items():
        in_path = DATA_DIR / filename
        rows = read_jsonl(in_path)
        if not rows:
            continue

        translated_rows = []
        for row in rows:
            out = dict(row)
            for field in fields:
                src = norm_text(out.get(field, ""))
                if src:
                    out[field] = cache[src]
            translated_rows.append(out)

        out_path = OUT_DIR / filename.replace(".jsonl", "_ro.jsonl")
        write_jsonl(out_path, translated_rows)
        outputs[filename] = str(out_path)
        print(f"Wrote {out_path} ({len(translated_rows)} rows)")

    return outputs


def build_quality_report(cache):
    flagged = []
    ratios = []
    for src, tgt in cache.items():
        flags, ratio = quality_flags(src, tgt)
        ratios.append(ratio)
        if flags:
            flagged.append(
                {
                    "flags": flags,
                    "length_ratio": round(ratio, 3),
                    "source_words": len(src.split()),
                    "target_words": len(tgt.split()),
                    "source_preview": src[:400],
                    "target_preview": tgt[:400],
                }
            )

    ratios_sorted = sorted(ratios)
    report = {
        "total_translations": len(cache),
        "flagged_count": len(flagged),
        "flagged_rate": len(flagged) / max(len(cache), 1),
        "length_ratio": {
            "min": min(ratios) if ratios else None,
            "median": ratios_sorted[len(ratios_sorted) // 2] if ratios_sorted else None,
            "max": max(ratios) if ratios else None,
        },
        "flag_counts": {},
        "flagged_examples": flagged[:200],
    }
    for item in flagged:
        for flag in item["flags"]:
            report["flag_counts"][flag] = report["flag_counts"].get(flag, 0) + 1

    QUALITY_REPORT_PATH.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"Saved quality report: {QUALITY_REPORT_PATH}")
    print(f"Flagged translations: {len(flagged)} / {len(cache)} ({report['flagged_rate'] * 100:.2f}%)")
    print("Flag counts:", report["flag_counts"])
    return report


def main():
    texts, file_stats = collect_unique_texts()
    cache = translate_all(texts)
    outputs = write_translated_files(cache)
    quality_report = build_quality_report(cache)

    manifest = {
        "model": MODEL_NAME,
        "source_language": SRC_LANG,
        "target_language": TGT_LANG,
        "batch_size": BATCH_SIZE,
        "max_input_tokens": MAX_INPUT_TOKENS,
        "max_new_tokens": MAX_NEW_TOKENS,
        "max_chunk_words": MAX_CHUNK_WORDS,
        "num_beams": NUM_BEAMS,
        "do_sample": False,
        "no_repeat_ngram_size": NO_REPEAT_NGRAM_SIZE,
        "repetition_penalty": REPETITION_PENALTY,
        "length_penalty": LENGTH_PENALTY,
        "force_retranslate": FORCE_RETRANSLATE,
        "cache_path": str(CACHE_PATH),
        "chunk_cache_path": str(CHUNK_CACHE_PATH),
        "quality_report_path": str(QUALITY_REPORT_PATH),
        "data_dir": str(DATA_DIR),
        "out_dir": str(OUT_DIR),
        "file_stats": file_stats,
        "outputs": outputs,
        "unique_texts": len(texts),
        "cached_translations": len(cache),
        "quality_summary": {
            "flagged_count": quality_report["flagged_count"],
            "flagged_rate": quality_report["flagged_rate"],
            "flag_counts": quality_report["flag_counts"],
        },
    }
    MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    print(f"Saved manifest: {MANIFEST_PATH}")


if __name__ == "__main__":
    main()
